<a href="https://colab.research.google.com/github/shumailkhan845/CV-lab-5/blob/main/CV_Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load DataSet

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sovitrath/neu-steel-surface-defect-detect-trainvalid-split")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'neu-steel-surface-defect-detect-trainvalid-split' dataset.
Path to dataset files: /kaggle/input/neu-steel-surface-defect-detect-trainvalid-split


In [2]:
DATA = path

Importing functions

In [ ]:
import os, re, time, glob, itertools
import numpy as np, cv2, pandas as pd, matplotlib.pyplot as plt, joblib
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay)

SEED, IMG = 42, 128
OUT = "lab05_output"; os.makedirs(OUT, exist_ok=True)

# Task 1: load (label = filename prefix, e.g. crazing_12.jpg -> crazing)
def load_dataset(root):
    paths = [p for e in ("jpg","jpeg","png","bmp")
             for p in glob.glob(os.path.join(root, "**", f"*.{e}"), recursive=True)]
    X, y = [], []
    for p in sorted(paths):
        m = re.match(r"([A-Za-z\-_]+?)_?\d+", os.path.basename(p))
        label = m.group(1).strip("_").lower() if m else os.path.basename(os.path.dirname(p)).lower()
        im = cv2.imread(p)
        if im is None: continue
        X.append(im); y.append(label)
    return X, np.array(y)

# Tasks 2-3: resize + grayscale
def preprocess(im):
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if im.ndim == 3 else im
    return cv2.resize(g, (IMG, IMG), interpolation=cv2.INTER_AREA)

# Task 4: HOG
def hog_feat(g, cell=8, ori=9):
    return hog(g, orientations=ori, pixels_per_cell=(cell, cell),
               cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)
def extract(grays, cell=8, ori=9):
    return np.array([hog_feat(g, cell, ori) for g in grays])

def metrics(yt, yp):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    return dict(accuracy=accuracy_score(yt, yp), precision=p, recall=r, f1=f)

def make_models():
    return {
        "HOG+SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale",
                          class_weight="balanced", probability=True, random_state=SEED)),
        "HOG+RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                          n_jobs=-1, random_state=SEED),
    }

# Task 12: distortions
def brightness(g, beta=40): return np.clip(g.astype(int)+beta, 0, 255).astype(np.uint8)
def noise(g, sigma=15):
    r = np.random.default_rng(SEED); return np.clip(g + r.normal(0, sigma, g.shape), 0, 255).astype(np.uint8)
def rotate(g, ang=15):
    M = cv2.getRotationMatrix2D((g.shape[1]/2, g.shape[0]/2), ang, 1.0)
    return cv2.warpAffine(g, M, g.shape[::-1], borderMode=cv2.BORDER_REFLECT)
def blur(g, k=5): return cv2.GaussianBlur(g, (k, k), 0)

# Task 13: final quality-control decision module
def inspect_image(img_bgr, bundle):
    g = preprocess(img_bgr)
    f = hog_feat(g, bundle["cell"], bundle["ori"]).reshape(1, -1)
    proba = bundle["model"].predict_proba(f)[0]; k = int(np.argmax(proba)); conf = proba[k]*100
    names = bundle["names"]
    if bundle["binary"]:
        defective = names[k] == "DEFECTIVE"; label = names[k]
    else:   # NEU-DET has no normal class -> every class is a defect type
        defective = True; label = f"DEFECTIVE ({names[k]})"
    print("PRODUCT INSPECTION RESULT")
    print(f"Prediction: {label}\nConfidence: {conf:.1f}%")
    print("Action: REJECT PRODUCT" if defective else "Action: ACCEPT PRODUCT")
    return defective, conf

Preprosessing and splitting the data

In [ ]:
X_raw, y_raw = load_dataset(DATA)
assert len(X_raw) > 0, "No images found - check the zip contents"
classes = sorted(set(y_raw))
print("Total images:", len(X_raw)); print(pd.Series(y_raw).value_counts())
pd.Series(y_raw).value_counts().to_csv(f"{OUT}/class_distribution.csv")

has_normal = any(c in ("normal","non_defective","non-defective") for c in classes)
if has_normal:
    y_lab = np.array([0 if c in ("normal","non_defective","non-defective") else 1 for c in y_raw])
    names = ["NON-DEFECTIVE", "DEFECTIVE"]
else:
    names = classes; y_lab = np.array([classes.index(c) for c in y_raw])

G = np.array([preprocess(im) for im in X_raw])

# single stratified 70/15/15 split (as in the paper)
idx = np.arange(len(G))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=y_lab, random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=y_lab[tmp], random_state=SEED)
print(f"train/val/test = {len(tr)}/{len(va)}/{len(te)}")

Visualizing the HOG

In [ ]:
show = classes[:6]
fig, ax = plt.subplots(2, len(show), figsize=(3*len(show), 6), squeeze=False)
for j, c in enumerate(show):
    i = np.where(y_raw == c)[0][0]
    _, vis = hog(G[i], orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2), visualize=True)
    ax[0,j].imshow(G[i], cmap="gray"); ax[0,j].set_title(c); ax[0,j].axis("off")
    ax[1,j].imshow(vis, cmap="gray");  ax[1,j].set_title("HOG"); ax[1,j].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/hog_visualisation.png", dpi=150); plt.show()

Comparing the SVM and Random Forest

In [ ]:
Ftr, Fva, Fte = (extract(G[s]) for s in (tr, va, te))
rows = []
for name, model in make_models().items():
    t0 = time.time(); model.fit(Ftr, y_lab[tr]); t_fit = time.time()-t0
    t0 = time.time(); pred = model.predict(Fte); t_inf = (time.time()-t0)/len(te)*1000
    m = metrics(y_lab[te], pred); m.update(model=name, train_s=t_fit, infer_ms_per_img=t_inf); rows.append(m)
    print(f"\n=== {name} ===\n", classification_report(y_lab[te], pred, target_names=names, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_lab[te], pred), display_labels=names).plot(xticks_rotation=45, cmap="Blues")
    plt.title(name); plt.tight_layout(); plt.savefig(f"{OUT}/cm_{name.split()[0].replace('+','_')}.png", dpi=150); plt.show()
comp = pd.DataFrame(rows); comp.to_csv(f"{OUT}/classifier_comparison.csv", index=False)
display(comp.round(4))

Analyzing HOG parameter

In [ ]:
grid = []
for cell, ori in itertools.product([4, 8, 16], [6, 9, 12]):
    t0 = time.time(); A, V, B = (extract(G[s], cell, ori) for s in (tr, va, te))
    t_ext = (time.time()-t0)/(len(tr)+len(va)+len(te))*1000
    mdl = make_models()["HOG+SVM (RBF)"].fit(A, y_lab[tr])
    m = metrics(y_lab[te], mdl.predict(B))
    m.update(cell=cell, orientations=ori, feat_dim=A.shape[1], extract_ms_per_img=t_ext,
             val_f1=metrics(y_lab[va], mdl.predict(V))["f1"])
    grid.append(m)
gdf = pd.DataFrame(grid); gdf.to_csv(f"{OUT}/hog_parameter_study.csv", index=False); display(gdf.round(4))
best = gdf.sort_values("val_f1", ascending=False).iloc[0]
bc, bo = int(best.cell), int(best.orientations)
print(f"Best HOG setting (validation F1): cell={bc}, orientations={bo}")

# final model (train+val) with the best HOG setting
trva = np.concatenate([tr, va])
final = make_models()["HOG+SVM (RBF)"].fit(extract(G[trva], bc, bo), y_lab[trva])
bundle = dict(model=final, cell=bc, ori=bo, names=names, binary=(names[0] == "NON-DEFECTIVE"))
joblib.dump(bundle, f"{OUT}/final_model.joblib")

Robustness test


In [ ]:
base = metrics(y_lab[te], final.predict(extract(G[te], bc, bo)))
rob = [dict(condition="clean", **base, d_acc=0.0, d_f1=0.0)]
tests = {"brightness +40": brightness, "brightness -40": lambda g: brightness(g, -40),
         "gaussian noise s=15": noise, "rotation 15deg": rotate, "blur 5x5": blur}
for k, fn in tests.items():
    m = metrics(y_lab[te], final.predict(extract(np.array([fn(g) for g in G[te]]), bc, bo)))
    rob.append(dict(condition=k, **m, d_acc=m["accuracy"]-base["accuracy"], d_f1=m["f1"]-base["f1"]))
rdf = pd.DataFrame(rob); rdf.to_csv(f"{OUT}/robustness.csv", index=False); display(rdf.round(4))

se = np.sqrt(base["accuracy"]*(1-base["accuracy"])/len(te))*100
print(f"Binomial SE of accuracy ~ {se:.2f} pts; differences under ~{2*se:.1f} pts are not meaningful (single run).")

plt.figure(figsize=(8,4)); plt.bar(rdf.condition, rdf.f1); plt.xticks(rotation=30, ha="right")
plt.ylabel("Macro F1"); plt.title("Robustness"); plt.tight_layout()
plt.savefig(f"{OUT}/robustness.png", dpi=150); plt.show()

Final inspection

In [ ]:
for i in np.random.default_rng(1).choice(te, 3, replace=False):
    print("True label:", y_raw[i])
    inspect_image(X_raw[i], bundle); print("-"*35)